# P&ID Stage 1 — Train on Colab (GPU)

**Luồng**: code từ GitHub (`git clone`) + data từ Google Drive (mount). Data KHÔNG đi qua git.

**Chuẩn bị 1 lần trước khi chạy notebook:**
1. Push code lên GitHub (xem `docs/PUSH_AND_COLAB.md`).
2. Upload thư mục `data/canonical/` (ảnh + `annotations.coco.json` + `digitizepid_splits.json`) lên Google Drive tại: `MyDrive/pid/data/canonical/`.

**Runtime**: Runtime → Change runtime type → **GPU (T4)**.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive (chứa data + nơi lưu checkpoint)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Sửa nếu bạn đặt data ở path khác trên Drive:
DRIVE_ROOT = '/content/drive/MyDrive/pid'
DATA_ROOT  = f'{DRIVE_ROOT}/data'        # phải chứa canonical/
RUNS_DIR   = f'{DRIVE_ROOT}/runs'         # checkpoint sống qua các session
assert os.path.isdir(f'{DATA_ROOT}/canonical'), f'Không thấy {DATA_ROOT}/canonical — upload data lên Drive trước.'
print('canonical images:', len(os.listdir(f'{DATA_ROOT}/canonical/images')))

## 3. Clone code từ GitHub + cài deps (uv)

In [ ]:
REPO_URL = 'https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git'  # <-- sửa
%cd /content
![ -d pid ] || git clone $REPO_URL pid
%cd /content/pid
!git pull
!pip install -q uv
# Colab đã có torch CUDA; --no-install-project để khỏi build wheel, chỉ cài deps.
!uv pip install --system -q ultralytics sahi ensemble-boxes albumentations timm hydra-core omegaconf pycocotools opencv-python-headless pillow pandas scikit-learn matplotlib tqdm pyyaml

## 4. Cấu hình đường dẫn

`src/pipeline` + `src/utils` là flat layout — thêm `src/` vào PYTHONPATH. Data đọc từ Drive, checkpoint ghi ra Drive.

In [ ]:
import os
os.environ['PYTHONPATH'] = '/content/pid/src'
os.environ['PID_DATA_ROOT'] = DATA_ROOT
os.environ['PID_CONFIG_DIR'] = '/content/pid/configs'
print('PID_DATA_ROOT =', DATA_ROOT)
print('RUNS_DIR      =', RUNS_DIR)

## 5. Data prep — chạy MỘT LẦN (kết quả lưu trên Drive)

`classes` → `manifest` → `yolo` → `tiles`. Bước `tiles` cắt 500 ảnh 7168×4561 thành ~27k tile 1024px (nặng nhất, ~10-20 phút). Vì output nằm trên Drive (`derived/`), các session sau **bỏ qua cell này** — tiles đã sẵn.

> Nếu `data/derived/tiles/data.yaml` đã tồn tại trên Drive, skip cell này.

In [ ]:
import os
tiles_yaml = f'{DATA_ROOT}/derived/tiles/data.yaml'
if os.path.exists(tiles_yaml):
    print('Tiles đã có trên Drive, skip prep:', tiles_yaml)
else:
    !cd /content/pid && PYTHONPATH=src python src/run_pipeline.py prep --step classes data_root=$DATA_ROOT
    !cd /content/pid && PYTHONPATH=src python src/run_pipeline.py prep --step manifest data_root=$DATA_ROOT
    !cd /content/pid && PYTHONPATH=src python src/run_pipeline.py prep --step yolo data_root=$DATA_ROOT detector.label_level=coarse data.label_level=coarse
    !cd /content/pid && PYTHONPATH=src python src/run_pipeline.py prep --step tiles data_root=$DATA_ROOT data.label_level=coarse
    print('Tiles xong.')

In [ ]:
# Kiểm tra tiles
import glob
print('train tiles:', len(glob.glob(f'{DATA_ROOT}/derived/tiles/images/train/*.jpg')))
print('val tiles  :', len(glob.glob(f'{DATA_ROOT}/derived/tiles/images/val/*.jpg')))
!cat $DATA_ROOT/derived/tiles/data.yaml

## 6. Train Stage 1 detector (coarse, 8 nhóm)

**Phiên đầu**: train từ đầu. Đặt `epochs` tổng lớn; nếu Colab ngắt, phiên sau resume (cell 7). Checkpoint ghi vào `RUNS_DIR` trên Drive nên sống sót.

In [ ]:
!cd /content/pid && PYTHONPATH=src python src/run_pipeline.py detect --train \
    data_root=$DATA_ROOT run_dir=$RUNS_DIR \
    detector.label_level=coarse detector.epochs=100 detector.batch=16 detector.imgsz=1024

## 7. Resume train (phiên sau khi Colab ngắt)

Sửa `RESUME_DIR` = run dir phiên trước (in ra ở cuối cell 6, dạng `runs/2026-...`). Resume nạp `last.pt` + optimizer/LR/epoch, train tiếp tới `epochs` gốc — KHÔNG truyền lại epochs/aug.

In [ ]:
RESUME_DIR = f'{RUNS_DIR}/2026-XX-XX_XX-XX-XX'  # <-- sửa thành run dir phiên trước
!cd /content/pid && PYTHONPATH=src python src/run_pipeline.py detect --train \
    data_root=$DATA_ROOT run_dir=$RUNS_DIR \
    detector.resume=true detector.resume_dir=$RESUME_DIR

## 8. (Sau khi train xong) Inference + eval

Sửa `BEST` = đường dẫn `best.pt` (in ra cuối cell 6/7).

In [ ]:
BEST = f'{RUNS_DIR}/2026-XX-XX_XX-XX-XX/train/weights/best.pt'  # <-- sửa
!cd /content/pid && PYTHONPATH=src python src/run_pipeline.py detect --infer \
    data_root=$DATA_ROOT run_dir=$RUNS_DIR detector.ckpt=$BEST detector.label_level=coarse
!cd /content/pid && PYTHONPATH=src python src/run_pipeline.py evaluate \
    data_root=$DATA_ROOT run_dir=$RUNS_DIR eval.mode=det_only eval.split=val